In [ ]:
# In the name of GOD, the Most Gracious, the Most Merciful

In [ ]:
# Python == 3.7.6, tensorflow==2.5.0, pandas == 1.3.5, numpy == 1.19.5, sklearn == 0.24.2

In [ ]:
# AlertPro, CICID2017

In [ ]:
import numpy as np
import pandas as pd
import random
import h5py
import joblib
import time
import itertools

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model
from tensorflow.compat.v1.losses import sparse_softmax_cross_entropy

from collections import deque 

import csv
import json
from json import JSONEncoder
from statistics import mode

from scipy.spatial.distance import cdist

In [ ]:
############################ DQNAgent class ##########################################
class DQNAgent:
    def __init__(self, state_dim, action_dim, batch_size, gamma, epsilon, epsilon_min, epsilon_decay, epsilon_initial, tau, dueling, ddqn):
        # Environment parameters
        self.state_dim = state_dim
        self.action_dim = action_dim

        # DQN alternatives
        self.ddqn = ddqn
        self.dueling = dueling
        
        # Neural network models
        self.model = self.build_model()
        self.target_model = self.build_model()

        
        # initialise target_model's weights with model's weights
        self.target_model.set_weights(self.model.get_weights())
        
        # Hyperparameters
        self.batch_size = batch_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.epsilon_initial = epsilon_initial
        self.tau = tau

        

    def build_model(self):
        X_input = Input(shape=(self.state_dim,))
        X = Dense(64, input_shape= (self.state_dim,), activation="relu")(X_input)
        X = Dense(64, activation='relu')(X)
        X = Dense(32, activation='relu')(X)
    
        if self.dueling:
            state_value = Dense(1)(X)
            action_advantage = Dense(self.action_dim)(X)
            X = (state_value + (action_advantage - tf.math.reduce_mean(action_advantage, axis=1, keepdims=True)))
        else:
            X = Dense(self.action_dim, activation="linear")(X)
            
    
        model = Model(inputs = X_input, outputs = X)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
        return model
    
    def get_action(self, state):
        mu = np.random.uniform(0,1) 
        
        if mu <= self.epsilon: 
            return np.random.choice([0, 1])

        else:
            input = np.array(state)  # Convert to a numpy array
            if input.ndim == 1:  
                input = np.expand_dims(input, axis=0)  # Reshape the input to a 2D array: (1, state_dim)
            q_values = self.model.predict(input)

            
            return np.argmax(q_values[0])
    
    
    # A method to set the replay buffer
    def set_replay_buffer(self, replay_buffer):  
        self.replay_buffer = replay_buffer
    
        
    # Train the deep learning model with a sample from replayBuffer
    def trainNetwork(self):
        optimal_Q_values = []
        epoch_losses =[]
        
        
        if len(self.replay_buffer) < self.batch_size:
            return 0, 0, 0 
        
        # sampling minibatch from replay buffer
        minibatch = self.replay_buffer.sample(self.batch_size)
        states, actions, rewards, next_states, dones = minibatch
        
        # Convert states into a list of DQN inputs
        dqn_inputs = [state for state in states]
        
        # Convert next_states into a list of DQN inputs
        next_dqn_inputs = [next_state for next_state in next_states]

        # Convert states and next_states into 2D numpy arrays
        dqn_inputs = np.array(dqn_inputs)  # Shape: (batch_size, state_dim)
        next_dqn_inputs = np.array(next_dqn_inputs)  # Shape: (batch_size, state_dim)


        
        target = self.model.predict(dqn_inputs)
        target_next = self.model.predict(next_dqn_inputs)
        target_val = self.target_model.predict(next_dqn_inputs)


        for i in range(self.batch_size):
            if dones[i]:
                target[i][actions[i]] = rewards[i]    
            else:
                if self.ddqn: # Double - DQN  # Choose DQN or Double DQN using self.ddqn
                    a = np.argmax(target_next[i])  
                    target[i][actions[i]] = rewards[i] + self.gamma * target_val[i][a] 
                else: # Standard - DQN
                    target[i][action[i]] = reward[i] + self.gamma * (np.amax(target_next[i]))

            optimal_Q_values.append(target[i][actions[i]])
            
            
        mean_q_value = np.mean(optimal_Q_values)
        max_q_value = np.max(optimal_Q_values)
                

        history = self.model.fit(dqn_inputs, target, epochs=50, verbose=0)
        epoch_losses.extend(history.history['loss'])

        return mean_q_value, max_q_value, epoch_losses
        
    def Q_value_calculate(self, state, action, reward, next_state, done):
        # Convert states into a list of DQN inputs
        dqn_input = np.array(state)
        dqn_input = np.expand_dims(dqn_input, axis=0)
        
        # Convert next_states into a list of DQN inputs
        next_dqn_input = np.array(next_state)
        next_dqn_input = np.expand_dims(next_dqn_input, axis=0)


        
        target = self.model.predict(dqn_input)
        target_next = self.model.predict(next_dqn_input)
        target_val = self.target_model.predict(next_dqn_input)
        
        
        if done:
            Q = reward   
        else:
            if self.ddqn: # Double - DQN
                a = np.argmax(target_next[0]) 
                Q = reward + self.gamma * target_val[0][a] 
            else: # Standard - DQN
                Q = reward + self.gamma * (np.amax(target_next[0]))

        return Q
        
        
        
    
    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())

    def update_target_model_soft(self):
        target_model_weights = self.target_model.get_weights()
        model_weights = self.model.get_weights()
        new_weights = []
        for target_model_weight, model_weight in zip(target_model_weights, model_weights):
            new_weight = self.tau * model_weight + (1 - self.tau) * target_model_weight
            new_weights.append(new_weight)
        self.target_model.set_weights(new_weights)
        

    def update_epsilon(self, total_time):
        if self.epsilon > self.epsilon_min:
            self.epsilon = self.epsilon_initial / (1+total_time*self.epsilon_decay)

In [ ]:
################# Replay buffer class #########################################
class ReplayBuffer:
    def __init__(self, max_size):
        self.max_size = max_size
        self.buffer = deque(maxlen=max_size)

    def push(self, experience):
        self.buffer.append(experience)
        
    def __len__(self):
        return len(self.buffer)

    def sample(self, batch_size):
        if len(self.buffer) < batch_size:
            batch_size = len(self.buffer)

        batch = random.sample(self.buffer, batch_size)
        state_batch, action_batch, reward_batch, next_state_batch, done_batch = [], [], [], [], []

        for experience in batch:
            state, action, reward, next_state, done = experience
            state_batch.append(state)
            action_batch.append(action)
            reward_batch.append(reward)
            next_state_batch.append(next_state)
            done_batch.append(done)

        return state_batch, action_batch, reward_batch, next_state_batch, done_batch

In [ ]:
def Reward(alert_DQN, top_threshold, bottom_threshold):
    
    if alert_DQN['revised_priority'] == 1: # True Positive (DRL decides the alert is an attack and sends it to the analyst, the analyst also verifies the alert as an attck)
        if alert_DQN['anomaly_score'] >= top_threshold:
            r = alert_DQN['anomaly_score']

        else:
            r = 1

            
    if alert_DQN['revised_priority'] == 0: # False Positive (DRL decides the alert is an attack and sends it to the analyst, the analyst considers the alert as Normal)
        if (alert_DQN['anomaly_score'] > bottom_threshold) and (alert_DQN['anomaly_score'] <= top_threshold):
            r = -(0.1 * alert_DQN['anomaly_score'])

        elif alert_DQN['anomaly_score'] <= bottom_threshold:
            r = -0.1

        else:
            r = -0.1
        
            
    return r 

In [ ]:
# A function to add alert to storages
def add_alert_to_storage(alert_DQN):
    columns = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12']
    
    # Create a new DataFrame row from alert['Attribute']
    new_row = pd.DataFrame([alert_DQN['Attribute']], columns=columns)
    
    # Define the mapping of priority to file paths
    storage_files = {
        0: 'Non_attack_storage.csv',
        1: 'Attack_storage.csv'
    }
    
    
    # Get the storage file based on the revised priority
    storage_file = storage_files.get(alert_DQN['revised_priority'])

       
    if storage_file:
        # Append the new row directly to the file without a header
        new_row.to_csv(storage_file, index=False, header=False, mode='a')


In [ ]:
    
def step(alert_DQN, state, action, step_length, index, investigation_time_alert, top_threshold, bottom_threshold): 
                                                                       
                                                                      
    if action == 1: # present alert to analyst

        alert_DQN['revised_priority'] = alert_DQN['priority_Analyst_label']  # this is the presentation to the analyst

        
        
        alert_DQN['allocated_time'] = investigation_time_alert
        
    
        reward = Reward(alert_DQN, top_threshold, bottom_threshold)

        
        # update Storages 

        add_alert_to_storage(alert_DQN) 
        
        # next state
        next_state = state_build(alert_DQN)
                                                                                                     
        
    elif action == 0: # Not present alert to analyst
        

                
        alert_DQN['allocated_time'] = 0
        
        reward = 0  # the alert is not presented to the analyst 
        

        
        # next state
        next_state = alert_DQN['alert_state'].copy()  
        

        
    if index < step_length - 1:
        return alert_DQN, reward, next_state, False
    else:
        return alert_DQN, reward, next_state, True

In [ ]:
def state_build(alert):
    
    # Function to calculate Euclidean distance
    def mean_euclidean_distance(new_alert, storage_df):
        # Ensure new_alert is a NumPy array with the correct shape
        new_alert = np.array(new_alert).reshape(1, -1)
        # Convert the storage DataFrame to a NumPy array for computation
        storage_array = storage_df.values
        # Compute pairwise distances between new_alert and all rows in storage_array
        distances = cdist(new_alert, storage_array, metric='euclidean')
        distances = np.round(distances, 6)
        # Calculate and return the mean of the distances
        mean_distance = np.mean(distances)
        mean_distance = np.round(mean_distance, 6)
        return mean_distance, distances
    
    # Reading storages from .csv files to DataFrames
    Attack_storage = pd.read_csv('Attack_storage.csv')
    Non_attack_storage = pd.read_csv('Non_attack_storage.csv')
    
    # Extract the 'Attribute' list from the alert_dict
    alert_features = alert['Attribute']

    # Column names for the PCA features
    pca_columns = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12']
    
    #@@@@@@@@ State (1) anomaly_score @@@@@@@@@@@@@@@@@@@@@@@@@
    state_1 = alert['anomaly_score']
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

    # Check the P1 and P2 features in Attack_storage

    # Check if the first PCA feature of the new alert is among the first PCA feature column of df
    P1_in_Attack_storage = alert_features[0] in Attack_storage['PC1'].values

    # Check if the second PCA feature of the new alert is among the second PCA feature column of df
    P2_in_Attack_storage = alert_features[1] in Attack_storage['PC2'].values

    # Check the P1 and P2 features in Non_attack_storage

    # Check if the first PCA feature of the new alert is among the first PCA feature column of df
    P1_in_Non_attack_storage = alert_features[0] in Non_attack_storage['PC1'].values

    # Check if the second PCA feature of the new alert is among the second PCA feature column of df
    P2_in_Non_attack_storage = alert_features[1] in Non_attack_storage['PC2'].values

    # P1 is related to Feature_12 and Feature_13
    Feature_12 = 0
    Feature_13 = 0

    if P1_in_Attack_storage == True:
        Feature_12 = 1
    else:
        Feature_12 = 0

    if P1_in_Non_attack_storage == True:
        Feature_13 = 1
    else:
        Feature_13 = 0


    #@@@@@@@@ State (2) P1, Feature 12 @@@@@@@@@@@@@@@@@@@@@@@@@
    state_2 = Feature_12
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

    #@@@@@@@@ State (3) P1, Feature 13 @@@@@@@@@@@@@@@@@@@@@@@@@
    state_3 = Feature_13
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@


    # P2 is related to Feature_14 and Feature_15
    Feature_14 = 0
    Feature_15 = 0

    if P2_in_Attack_storage == True:
        Feature_14 = 1
    else:
        Feature_14 = 0

    if P2_in_Non_attack_storage == True:
        Feature_15 = 1
    else:
        Feature_15 = 0


    #@@@@@@@@ State (4) P2, Feature 14 @@@@@@@@@@@@@@@@@@@@@@@@@
    state_4 = Feature_14
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

    #@@@@@@@@ State (5) P2, Feature 15 @@@@@@@@@@@@@@@@@@@@@@@@@
    state_5 = Feature_15
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@


    alert_attributes = np.array(alert['Attribute'])


    # Euclidean Distance to storages
    mean_Attack_distance, Attack_distances = mean_euclidean_distance(alert_attributes, Attack_storage)
    mean_Non_Attack_distance, Non_Attack_distances = mean_euclidean_distance(alert_attributes, Non_attack_storage)
    


    # Calculate Features 16 and 17 of AlertPro paper 
    # two state itms of DRL based on the nearst N=5 alerts form Attack_storage and Non_attack_storage to the alert

    # Step 1: Flatten
    Attack_distances_flatten = Attack_distances.flatten()
    Non_Attack_distances_flatten = Non_Attack_distances.flatten()

    # Step 2: Labeling
    attack_labeled = [(d, 'attack') for d in Attack_distances_flatten]
    non_attack_labeled = [(d, 'non_attack') for d in Non_Attack_distances_flatten]

    # Step 3: Combine
    combined_labeled = np.concatenate([attack_labeled, non_attack_labeled])

    # Step 4: Sort
    combined_labeled_sorted = sorted(combined_labeled, key=lambda x: x[0])

    # Step 5: Take top 5 (N=5 based on AlertPro paper)
    top5 = combined_labeled_sorted[:5]

    # Step 6: Count
    Attack_distances_count = sum(1 for d, label in top5 if label == 'attack')
    Non_Attack_distances_count = sum(1 for d, label in top5 if label == 'non_attack')
    



    if Attack_distances_count > 0:
        Feature_16 = 1
    else:
        Feature_16 = 0    

    #@@@@@@@@ State (6) nearst N=5 alerts form Attack_storage @@@@@@@@@@@@@@@@@@@@@@@@@
    state_6 = Feature_16
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

    if Non_Attack_distances_count > 0:
        Feature_17 = 1
    else:
        Feature_17 = 0    

    #@@@@@@@@ State (7) nearst N=5 alerts form Non_attack_storage @@@@@@@@@@@@@@@@@@@@@@@@@
    state_7 = Feature_17
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

    Feature_18 = mean_Attack_distance

    #@@@@@@@@ State (8) mean Euclidean distance to Attack_storage @@@@@@@@@@@@@@@@@@@@@@@@@
    state_8 = Feature_18
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

    Feature_19 = mean_Non_Attack_distance

    #@@@@@@@@ State (9) mean Euclidean distance to Non_attack_storage @@@@@@@@@@@@@@@@@@@@@
    state_9 = Feature_19
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

    # alert state 
    state = [state_1, state_2, state_3, state_4, state_5, state_6, state_7, state_8, state_9]
            
    return state

In [ ]:
def AP(agent, replay_buffer, num_episodes, num_time_steps, Features, CVSS_df, rest_combined_alert_data):


    # Initialise an empty dictionary to store metrics
    metrics_data = {}
    size_alert_left = len(Features)
    lower_bound = 0

    
    # Load Isolation Forest model
    Isolation_Forest = joblib.load('iso_forest_CICIDS_ThreeSeven.joblib')
    
    for episode in range(num_episodes):

        episode_key = f"episode_{episode + 1}"  # Use a dynamic key for each episode
        
        # Initialise an empty dictionary for the current episode metrics
        episode_metrics = {}

        # Poisson distribution
        # Mean number of alerts per hour (λ)
        lambda_ = 400
        
        # Number of time steps (hours)
        steps = num_time_steps  # 24
        
        # Generate the number of alerts for each time step using Poisson distribution
        alerts_number_all_time_steps = np.random.poisson(lambda_, steps)
        
        
        for time_step in range(num_time_steps):
            
            start_time = time.time()  # Record the start time
            
            time_step_key = f"time_step_{time_step + 1}"  # Use a dynamic key for each time step
            
            # for my clarification
            time_step = time_step 

            # The total time of run so far
            total_time = (episode * num_time_steps) + time_step

            ########## Alert selection for each time_step ######################################################################
            
            # The number of input alerts to SOC based on Poisson distribution  
            alerts_per_step = alerts_number_all_time_steps[time_step]

                   
            # Determine how many alerts to select
            alerts_to_select = min(alerts_per_step, size_alert_left)


            upper_bound = lower_bound + alerts_to_select
            
            
            # Select the next batch of alerts
            selected_alert_data = Features.iloc[lower_bound : upper_bound]
            
        
            # Select the reprioritisation values of alerts      
            selected_reprioritisation_data = CVSS_df[lower_bound : upper_bound]
            

            selected_combined_alert_data = rest_combined_alert_data[lower_bound : upper_bound]
            
            
            # alerts for each time_step
            size_alert_left = size_alert_left - len(selected_alert_data)
            lower_bound = upper_bound
            
            ###############################################################################################################                              
            ################ Ensemble prioritisation ######################################################################

            X_new = selected_alert_data.to_numpy()
            
            CVSS_label = selected_reprioritisation_data['CVSS_value']            # Can be used for the analyst
            Identifiers = selected_reprioritisation_data['Identifier'].tolist()
            Attack_types = selected_reprioritisation_data[' Attack category'].tolist()
            Label = selected_reprioritisation_data['Label'].tolist()   # 0 or 1         # Can be used for the analyst
            

            

            # Calculating labels using isolation forst
            # This is not done by AlertPro paper. I do it because I need these valuse to calculate the accuracy of resluts
            IF_labels = Isolation_Forest.predict(X_new) # 1 or -1
            IF_labels = IF_labels.tolist() # 1 or -1
            

        
            # Calculating anomaly scores using isolation forst
            opposite_IF_scores = Isolation_Forest.score_samples(X_new) # between -1 and 0
            IF_scores = - opposite_IF_scores  # between 0 and 1
            IF_scores = np.round(IF_scores, 6) # roand the values to 6 digits
            

            
            # Ensure IF_scores are numpy arrays
            IF_scores = np.array(IF_scores)
            
            ####### Inputs for rewrad calculation ###################
            # Sort (descending)
            sorted_IF_scores = np.sort(IF_scores)[::-1]
            

            
            NN = 19 # N threshold in AlertPro paper (Equation 2) # It is 19 in our case because in the training dataset 19% of alerts are attack (anomaly)
            Length_sorted_IF_scores = len(sorted_IF_scores)
            


            top_k = int(np.ceil(NN / 100 * (Length_sorted_IF_scores)))   # e.g., 11% of 410 ≈ 46
            

            

            
            bottom_k = top_k

            
            middle_end = Length_sorted_IF_scores - bottom_k

            

            
            top_threshold = sorted_IF_scores[top_k - 1]

            
            bottom_threshold = sorted_IF_scores[middle_end]

            
            ######## Building the state dimensions for each alert #########

            alert_list =[]
            # Convert predicted_alerts (prioritised alerts) into a list of dictionaries with required features
            for index, value in enumerate(IF_scores):
                
                alert_id = f'{total_time}_{index+1}'
                

                # Create the dictionary for each alert
                alert_dict = {
                    'alert_id': alert_id,
                    'Identifier': Identifiers[index],
                    'Attack_type': Attack_types[index],
                    'anomaly_score': value, 
                    'IF_label': IF_labels[index],
                    'priority_Analyst': CVSS_label.iloc[index],
                    'priority_Analyst_label': Label[index],
                    'Attribute': selected_alert_data.iloc[index].tolist(),
                    'revised_priority': 10,
                    'allocated_time': None,
                    'alert_state': None,
                    'alert_action': None,
                    'alert_reward': None,
                    'alert_next_state': None,
                }


                # Append the dictionary to the list
                alert_list.append(alert_dict)
                
            
            
            # Here, we include Normal alerts. 
            # But, to avoid changing the name of variables and the code we use the name 'alert_list_without_Normal'
            alert_list_without_Normal = alert_list

            

            # lists to collect alerts whose 'revised_priority' are assigned and are not assigned
            alert_list_without_Normal_with_revised_priority = []
            alert_list_without_Normal_without_revised_priority = []

            for alert in alert_list_without_Normal:
                
                if alert['revised_priority'] != 10:
                    alert_list_without_Normal_with_revised_priority.append(alert)
                else: 
                    # Add the alert to the list of without revised priority to send to DQN
                    alert_list_without_Normal_without_revised_priority.append(alert)
                
 ################# Interacting DQN with analyst ########################   
           

            ## We do not sort alerts based on their Ensemble priority to present to DQN, because this is a part of contribution of L2DHF, and other models do not generally do this
            ## We just keep the name 'sorted_alert_list_without_Normal_without_revised_priority' to be consistent with L2DHF code and do not require to change the rest of code   
            sorted_alert_list_without_Normal_without_revised_priority = alert_list_without_Normal_without_revised_priority
                        
            
            step_length = len(sorted_alert_list_without_Normal_without_revised_priority)
            analyst_time_percentage_for_analysis = 0.8
            remaining_time = int(60 * 60 * analyst_time_percentage_for_analysis) # analyst available time to review alerts in a time step (in seconds)
            

            # Lists of performance
            alerts_after_action_015 = []
            alerts_after_action_01 = [] # this gathers alerts after DQN decision (0: accept, 1: defer)
            reward_list = []
            Q_value_each_state_list = []
            mean_q_value_list = []
            max_q_value_list = []
            epoch_losses_list = []
            mean_target_list = []

            def calculate_average(list_A):
                if len(list_A) == 0:
                    return 0
                return sum(list_A) / len(list_A)
            
            ## Since in the original paper labels (-1: attck, 1: normal) are not used to determine investigation times, we do not determine them similar to the approach in L2DHF
            ## We set investigation times as the average of investigation times for different alert categories (150)
            investigation_time_alert = 150 
            
            for index, alert_DQN in enumerate(sorted_alert_list_without_Normal_without_revised_priority):


                if remaining_time < investigation_time_alert:
                    print(f"Loop terminated with remaining time: {remaining_time}")
                    break

                else: # DQN starts 
                    # state
                    #@@@@@@@@ alert state @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
                    alert_DQN['alert_state'] = state_build(alert_DQN)
                    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

                    state = alert_DQN['alert_state'].copy()


                    # action
                    action = agent.get_action(state)

                    

                    # step
                    alert_DQN, reward, next_state, done = step(alert_DQN, state, action, step_length, index, investigation_time_alert, top_threshold, bottom_threshold)


                    alert_DQN['alert_action'] = action
                    alert_DQN['alert_reward'] = reward
                    alert_DQN['alert_next_state'] = next_state
                    

                    # Store the experience in the replay buffer
                    replay_buffer.push((state, action, reward, next_state, done))

                    # train the DQN network
                    mean_q_value, max_q_value, epoch_losses = agent.trainNetwork()


                    #soft update the target network
                    agent.update_target_model_soft()

                    # calculate Q value of this experience (each state) (optional-not important)
                    Q_value = agent.Q_value_calculate(state, action, reward, next_state, done)




                    # update performance lists 
                    reward_list.append(reward)
                    mean_q_value_list.append(mean_q_value)
                    max_q_value_list.append(max_q_value)
                    Q_value_each_state_list.append(Q_value)
                    epoch_losses_list.append(epoch_losses)
                    alerts_after_action_01.append(alert_DQN)


                    # update remaining time
                    remaining_time -= investigation_time_alert
                        
                    

                

                        
            end_time = time.time()  # Record the end time
            time_step_execution_time = end_time - start_time  # Calculate the duration of execution of this loop         
                    
            
            # Update the epsilon value for epsilon-greedy exploration each time_step
            agent.update_epsilon(total_time)
                    
            # all alerts without Normal of the time step after actions
            Final_alert_list_without_Normal_addressed = list(itertools.chain(alert_list_without_Normal_with_revised_priority, alerts_after_action_01))
            reward_average = calculate_average(reward_list) 
            mean_q_value_average = calculate_average(mean_q_value_list)
            max_q_value_average = calculate_average(max_q_value_list)
            Q_value_each_state_average = calculate_average(Q_value_each_state_list)
            
            

            # Extract alert_ids from alerts_after_action_01
            alert_ids_after_action_01 = {alert['alert_id'] for alert in alerts_after_action_01}
            
            # Un_investigated alerts by analyst 
            Final_alert_list_without_Normal_NotAddressed = [
                alert for alert in alert_list_without_Normal_without_revised_priority
                if alert['alert_id'] not in alert_ids_after_action_01
            ]
            
            Analyst_Workload = len(Final_alert_list_without_Normal_NotAddressed)
            
            # Store metrics in the current time step
            current_metrics = {
                "Number_total_alerts": len(selected_alert_data),
                "Mean_Q_values": mean_q_value_average,
                "Max_Q_values": max_q_value_average,
                "Q_value_each_state_average": Q_value_each_state_average,
                "Q_value_each_state_list": Q_value_each_state_list,
                "Epoch_Losses": epoch_losses_list,
                "Reward": reward_average,
                "Reward_list": reward_list,
                "alert_list_without_Normal": alert_list_without_Normal,
                "alert_list_without_Normal_with_revised_priority": alert_list_without_Normal_with_revised_priority, 
                "alert_list_without_Normal_without_revised_priority": alert_list_without_Normal_without_revised_priority, 
                "alerts_after_action_01": alerts_after_action_01,
                "Final_alert_list_without_Normal_addressed": Final_alert_list_without_Normal_addressed,
                "Final_alert_list_without_Normal_NotAddressed": Final_alert_list_without_Normal_NotAddressed,
                "time_step_execution_time": time_step_execution_time,
                "Analyst_Workload": Analyst_Workload,
                "Remaining_time": remaining_time
            }
            
            episode_metrics[time_step_key] = current_metrics
            
            class NumpyEncoder(json.JSONEncoder):
                def default(self, obj):
                    if isinstance(obj, np.ndarray):
                        return obj.tolist()
                    elif isinstance(obj, np.integer):
                        return int(obj)
                    elif isinstance(obj, np.floating):
                        return float(obj)
                    return super(NumpyEncoder, self).default(obj)

            output_folder = 'C:\\...\\AlertPro_CICIDS2017\\'
            
            # Save the dictionary of metrics to a JSON file
            with open(f'{output_folder}AlertPro_CICIDS2017_time_step_{total_time + 1}.json', 'w') as outfile:
                json.dump(current_metrics, outfile, cls=NumpyEncoder)
                
            

            
            

        # Store the episode metrics in the main dictionary
        metrics_data[episode_key] = episode_metrics
        
        
    return metrics_data

In [ ]:
############################################### Load the alert dataset ######################################################################
################ pca_12_one_hot
#### subset 1, 2, 3, 4, 5 
df_PCA_12_combined_data_CIC_reduced_normalised_1 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_1.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_2 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_2.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_3 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_3.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_4 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_4.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_5 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_5.csv')

In [ ]:
# Shuffle data
df_PCA_12_combined_data_CIC_reduced_normalised_1 = df_PCA_12_combined_data_CIC_reduced_normalised_1.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_2 = df_PCA_12_combined_data_CIC_reduced_normalised_2.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_3 = df_PCA_12_combined_data_CIC_reduced_normalised_3.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_4 = df_PCA_12_combined_data_CIC_reduced_normalised_4.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_5 = df_PCA_12_combined_data_CIC_reduced_normalised_5.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# Combine 5 sub_datasets to a dataset
combined_alert_data = pd.concat([df_PCA_12_combined_data_CIC_reduced_normalised_1, df_PCA_12_combined_data_CIC_reduced_normalised_2, df_PCA_12_combined_data_CIC_reduced_normalised_3, df_PCA_12_combined_data_CIC_reduced_normalised_4, df_PCA_12_combined_data_CIC_reduced_normalised_5], ignore_index=True)

In [ ]:
# Shuffle combined_alert_data (second shuffle)
combined_alert_data = combined_alert_data.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_1

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_2

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_3

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_4

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_5

In [ ]:
combined_alert_data

In [ ]:
############# Building the Storages #################

In [ ]:
# Number of CVSS categories in the dataset
cvss_counts = combined_alert_data['CVSS'].value_counts()
cvss_counts

In [ ]:
# Building the storages of Critical, High, Medium, Low, and Normal
# Set the desired sample sizes
sample_sizes = {
    'Critical': 5,
    'High': 5,
    'Moderate': 5,
    'Low': 0,
    'Normal': 5
}

# Initialise a dictionary to hold the sampled df
sampled_dfs = {}

# Initialise a list to collect the indices of sampled rows
sampled_indices = []

# Iterate over the sample sizes dictionary
for cvss_value, sample_size in sample_sizes.items():
    # Filter the df for the current CVSS value
    filtered_combined_alert_data = combined_alert_data[combined_alert_data['CVSS'] == cvss_value]
    
    # Randomly sample the specified number of rows
    sampled_df = filtered_combined_alert_data.sample(n=sample_size, random_state=42)
    
    
    sampled_dfs[cvss_value] = sampled_df
    
    
    sampled_indices.extend(sampled_df.index)

# Drop the sampled rows from combined_alert_data
combined_alert_data = combined_alert_data.drop(sampled_indices)

# Separate sets for each CVSS value 
critical_set = sampled_dfs['Critical']
high_set = sampled_dfs['High']
moderate_set = sampled_dfs['Moderate']
low_set = sampled_dfs['Low']
normal_set = sampled_dfs['Normal']


In [ ]:
# Combine critical_set, high_set and moderate_set to obtain the attack_set 
Attack_set = pd.concat(
    [critical_set, high_set, moderate_set],
    axis=0,
    ignore_index=True
)

In [ ]:
normal_set

In [ ]:
Attack_set

In [ ]:
# Remove labels form storages
columns_to_remove = ['Label', 'CVSS', 'CVSS_value', 'Identifier', ' Attack category']

Non_attack_storage = normal_set.drop(columns=columns_to_remove, axis=1)
Attack_storage = Attack_set.drop(columns=columns_to_remove, axis=1)

In [ ]:
Non_attack_storage

In [ ]:
Attack_storage

In [ ]:
# Save df storages to .csv files
Non_attack_storage.to_csv('Non_attack_storage.csv', index=False)
Attack_storage.to_csv('Attack_storage.csv', index=False)

In [ ]:
# Remove data used for training IF from dataset to avoid data leakage 

In [ ]:
last_250k_combined_alert_data = combined_alert_data.iloc[-250000:]
rest_combined_alert_data = combined_alert_data.iloc[:-250000]

In [ ]:
last_250k_combined_alert_data

In [ ]:
rest_combined_alert_data

In [ ]:
######### Building alerts without labels for prioritisation and separate labels ############

In [ ]:
# Separating alert features and re-prioritisation columns 
columns_to_remove = ['Label', 'CVSS', 'CVSS_value', 'Identifier', ' Attack category']
Features = rest_combined_alert_data.drop(columns=columns_to_remove, axis=1)
CVSS_df = rest_combined_alert_data[['Label', 'CVSS', 'CVSS_value', 'Identifier', ' Attack category']]

In [ ]:
CVSS_df

In [ ]:
Features

In [ ]:
############################################## Main program ###########################################
if __name__ == "__main__":
    # state and action size
    state_dim = 9    
    action_dim = 2  
      
    # time parameters
    num_episodes = 7 * 12  # days of SOC working 12*7=84
    num_time_steps = 24  # hours of day for SOC working, # one working shift  
    
        
        
    # Hyperparameters
    max_size = 1000 # size of replay_buffer 
    batch_size = 64  # size of training sample from reply_buffer  
    gamma = 0.70    # Discount factor   
                                       
    epsilon = 0.75  # greedy parameter # the higher, the more degree of exploration versus exploitation 
    epsilon_initial = 0.75
    epsilon_min = 0.01
    epsilon_decay = 0.005
    ddqn = True
    dueling = True
    
    #for updating the target network
    tau = 0.001  
    
    
    # Instantiate the ReplayBuffer
    replay_buffer = ReplayBuffer(max_size)
    
    # Instantiate the DQNAgent
    agent = DQNAgent(state_dim, action_dim, batch_size, gamma, epsilon, epsilon_min, epsilon_decay, 
                     epsilon_initial, tau, dueling, ddqn)
    agent.set_replay_buffer(replay_buffer)
    
       
    # Run the model    
    metrics_data = AP(agent, replay_buffer, num_episodes, num_time_steps, Features, CVSS_df, rest_combined_alert_data)
    